In [10]:
import os
from google.colab import userdata
import torch.optim as optim
import wandb
import sys
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.nn.functional as F
import torch


In [11]:

os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
os.environ['GITHUB_TOKEN'] = userdata.get('GITHUB_TOKEN')
os.environ['GITHUB_USERNAME'] = userdata.get('GITHUB_USERNAME')

!git clone https://{os.environ['GITHUB_TOKEN']}@github.com/{os.environ['GITHUB_USERNAME']}/ML_hw_04_Facial-Expression-Recognition-Challenge.git
%cd ML_hw_04_Facial-Expression-Recognition-Challenge

!pip install -q wandb kaggle


wandb.login(key=os.environ['WANDB_API_KEY'])

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/access_token', 'w') as f:
    f.write(os.environ['KAGGLE_KEY'])

os.makedirs('/content/data', exist_ok=True)
!kaggle competitions download -c challenges-in-representation-learning-facial-expression-recognition-challenge -p /content/data
!unzip -q -o /content/data/*.zip -d /content/data

sys.path.append('/content/ML_hw_04_Facial-Expression-Recognition-Challenge/src')
from dataset import FERDataset, load_data, EMOTION_LABELS

print("Setup complete!")

Cloning into 'ML_hw_04_Facial-Expression-Recognition-Challenge'...
remote: Enumerating objects: 48, done.
remote: Counting objects: 100% (48/48), done.
remote: Compressing objects: 100% (40/40), done.
remote: Total 48 (delta 17), reused 10 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (48/48), 99.47 KiB | 8.29 MiB/s, done.
Resolving deltas: 100% (17/17), done.
/content/ML_hw_04_Facial-Expression-Recognition-Challenge/ML_hw_04_Facial-Expression-Recognition-Challenge


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


challenges-in-representation-learning-facial-expression-recognition-challenge.zip: Skipping, found more recently modified local copy (use --force to force download)
Setup complete!


In [12]:
from torch.utils.data import DataLoader

train, val, test=load_data('/content/data/train.csv')
print("Train:", train.shape, "Val:", val.shape, "Test:", test.shape)

train_data=FERDataset(train)
val_data=FERDataset(val)
test_data=FERDataset(test)

BATCH_SIZE=64
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False)

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))

Train: (20095, 2) Val: (4307, 2) Test: (4307, 2)
Train batches: 314
Val batches: 68
Test batches: 68


In [13]:
class DeeperCNN(nn.Module):
    def __init__(self, num_classes=7, dropout_p=0.0):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(128 * 6 * 6, 512)
        self.dropout = nn.Dropout(dropout_p)
        self.fc2 = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

In [20]:
import torch.optim as optim
import copy

def train_model(model, train_loader, val_loader, config, group_name, run_name):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()

    if config.get('optimizer', 'Adam') == 'SGD':
        optimizer = optim.SGD(model.parameters(), lr=config['learning_rate'], momentum=0.9)
    else:
        optimizer = optim.Adam(model.parameters(), lr=config['learning_rate'],
                                 weight_decay=config.get('weight_decay', 0))
    scheduler = None
    if config.get('scheduler') == 'ReduceLROnPlateau':
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.5)

    best_val_loss = float('inf')
    patience_counter = 0
    early_stop_patience = config.get('early_stop_patience', 4)
    best_model_wts = copy.deepcopy(model.state_dict())

    run = wandb.init(project="facial-expression-recognition", group=group_name, name=run_name, config=config)

    for epoch in range(config['epochs']):
        model.train()
        train_loss, train_correct, train_total = 0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            train_correct += (predicted == labels).sum().item()
            train_total += labels.size(0)

        train_loss /= train_total
        train_acc = train_correct / train_total

        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * images.size(0)
                _, predicted = torch.max(outputs, 1)
                val_correct += (predicted == labels).sum().item()
                val_total += labels.size(0)

        val_loss /= val_total
        val_acc = val_correct / val_total

        current_lr = optimizer.param_groups[0]['lr']

        wandb.log({"epoch": epoch+1, "train_loss": train_loss, "train_acc": train_acc,
                   "val_loss": val_loss, "val_acc": val_acc, "learning_rate": current_lr})
        print(f"Epoch {epoch+1}/{config['epochs']} | Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

        if scheduler is not None:
            scheduler.step(val_loss)


        if config.get('early_stopping', False):
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                patience_counter = 0
                best_model_wts = copy.deepcopy(model.state_dict())
            else:
                patience_counter += 1
                if patience_counter >= early_stop_patience:
                    break


    if config.get('early_stopping', False):
        model.load_state_dict(best_model_wts)


    wandb.finish()
    return model

In [7]:
model_check=DeeperCNN()
sample_batch, sample_labels = next(iter(train_loader))
output = model_check(sample_batch)
print("Output shape:", output.shape)
criterion=nn.CrossEntropyLoss()
loss=criterion(output, sample_labels)
print("Loss:", loss.item(), "| Expected ~", torch.log(torch.tensor(7.0)).item())

Output shape: torch.Size([64, 7])
Loss: 1.9318790435791016 | Expected ~ 1.945910096168518


In [8]:
small_subset = train.sample(n=200, random_state=42)
small_dataset = FERDataset(small_subset)
small_loader = DataLoader(small_dataset, batch_size=32, shuffle=True)

config_sanity2 = {"architecture": "DeeperCNN-sanity", "learning_rate": 0.001, "batch_size": 32,
                   "epochs": 15, "optimizer": "Adam", "dropout": 0.0, "weight_decay": 0}

model_sanity2 = DeeperCNN(dropout_p=0.0)
model_sanity2 = train_model(model_sanity2, small_loader, small_loader, config_sanity2,
                             group_name="sanity-checks",
                             run_name="architecture2-overfit-200-samples")

Epoch 1/15 | Train Loss: 4.5718, Train Acc: 0.2100 | Val Loss: 1.9427, Val Acc: 0.1800
Epoch 2/15 | Train Loss: 2.5536, Train Acc: 0.2100 | Val Loss: 1.8400, Val Acc: 0.2250
Epoch 3/15 | Train Loss: 1.8212, Train Acc: 0.2450 | Val Loss: 1.7807, Val Acc: 0.2400
Epoch 4/15 | Train Loss: 1.6745, Train Acc: 0.3750 | Val Loss: 1.7640, Val Acc: 0.3050
Epoch 5/15 | Train Loss: 1.6771, Train Acc: 0.3450 | Val Loss: 1.6667, Val Acc: 0.3600
Epoch 6/15 | Train Loss: 1.5661, Train Acc: 0.3750 | Val Loss: 1.5890, Val Acc: 0.3600
Epoch 7/15 | Train Loss: 1.5255, Train Acc: 0.3900 | Val Loss: 1.5449, Val Acc: 0.3800
Epoch 8/15 | Train Loss: 1.4436, Train Acc: 0.4450 | Val Loss: 1.4200, Val Acc: 0.4900
Epoch 9/15 | Train Loss: 1.3773, Train Acc: 0.5000 | Val Loss: 1.3383, Val Acc: 0.5350
Epoch 10/15 | Train Loss: 1.3080, Train Acc: 0.4800 | Val Loss: 1.2779, Val Acc: 0.5250
Epoch 11/15 | Train Loss: 1.2685, Train Acc: 0.5300 | Val Loss: 1.2009, Val Acc: 0.5900
Epoch 12/15 | Train Loss: 1.2291, Train A

epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
train_acc,▁▁▁▃▃▃▃▄▅▅▅▅▆██
train_loss,█▄▃▃▃▂▂▂▂▂▂▂▁▁▁
val_acc,▁▂▂▃▃▃▄▅▅▅▆▇▇▇█
val_loss,█▇▇▇▆▆▅▅▄▄▃▂▂▂▁
epoch,15
train_acc,0.71
train_loss,0.87251
val_acc,0.735
val_loss,0.87695


In [9]:
config_a2_r1 = {"architecture": "DeeperCNN", "learning_rate": 0.001, "batch_size": 64,
                 "epochs": 20, "optimizer": "Adam", "dropout": 0.0, "weight_decay": 0}

model_a2_r1 = DeeperCNN(dropout_p=0.0)
model_a2_r1 = train_model(model_a2_r1, train_loader, val_loader, config_a2_r1,
                           group_name="architecture-2-deeper-cnn",
                           run_name="no-regularization")

Epoch 1/20 | Train Loss: 1.7322, Train Acc: 0.3337 | Val Loss: 1.4386, Val Acc: 0.4423
Epoch 2/20 | Train Loss: 1.3630, Train Acc: 0.4783 | Val Loss: 1.3224, Val Acc: 0.4866
Epoch 3/20 | Train Loss: 1.2363, Train Acc: 0.5254 | Val Loss: 1.2124, Val Acc: 0.5370
Epoch 4/20 | Train Loss: 1.1424, Train Acc: 0.5674 | Val Loss: 1.2031, Val Acc: 0.5373
Epoch 5/20 | Train Loss: 1.0588, Train Acc: 0.6025 | Val Loss: 1.1559, Val Acc: 0.5568
Epoch 6/20 | Train Loss: 0.9705, Train Acc: 0.6346 | Val Loss: 1.3099, Val Acc: 0.5147
Epoch 7/20 | Train Loss: 0.8817, Train Acc: 0.6689 | Val Loss: 1.2759, Val Acc: 0.5394
Epoch 8/20 | Train Loss: 0.7918, Train Acc: 0.7047 | Val Loss: 1.3374, Val Acc: 0.5312
Epoch 9/20 | Train Loss: 0.6753, Train Acc: 0.7513 | Val Loss: 1.2750, Val Acc: 0.5642
Epoch 10/20 | Train Loss: 0.5811, Train Acc: 0.7875 | Val Loss: 1.3659, Val Acc: 0.5577
Epoch 11/20 | Train Loss: 0.4659, Train Acc: 0.8305 | Val Loss: 1.4445, Val Acc: 0.5668
Epoch 12/20 | Train Loss: 0.3778, Train A

epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
train_acc,▁▃▃▄▄▄▅▅▆▆▆▇▇███████
train_loss,█▆▆▅▅▅▄▄▃▃▃▂▂▂▁▁▁▁▁▁
val_acc,▁▃▆▆▇▅▆▆█▇█▆▇██▆▇██▇
val_loss,▂▂▁▁▁▂▁▂▁▂▂▄▃▄▄▅▅▆▇█
epoch,20
train_acc,0.97014
train_loss,0.09556
val_acc,0.5512
val_loss,2.85567


In [15]:
config_a2_r2 = {"architecture": "DeeperCNN", "learning_rate": 0.001, "batch_size": 64,
                 "epochs": 20, "optimizer": "Adam", "dropout": 0.5, "weight_decay": 0}

model_a2_r2 = DeeperCNN(dropout_p=0.5)
model_a2_r2 = train_model(model_a2_r2, train_loader, val_loader, config_a2_r2,
                           group_name="architecture-2-deeper-cnn",
                           run_name="dropout-0.5")

Epoch 1/20 | Train Loss: 1.8411, Train Acc: 0.2707 | Val Loss: 1.6026, Val Acc: 0.3650
Epoch 2/20 | Train Loss: 1.5774, Train Acc: 0.3783 | Val Loss: 1.4199, Val Acc: 0.4411
Epoch 3/20 | Train Loss: 1.4747, Train Acc: 0.4242 | Val Loss: 1.3810, Val Acc: 0.4560
Epoch 4/20 | Train Loss: 1.4093, Train Acc: 0.4531 | Val Loss: 1.2984, Val Acc: 0.4969
Epoch 5/20 | Train Loss: 1.3603, Train Acc: 0.4671 | Val Loss: 1.2494, Val Acc: 0.5106
Epoch 6/20 | Train Loss: 1.3129, Train Acc: 0.4922 | Val Loss: 1.2444, Val Acc: 0.5250
Epoch 7/20 | Train Loss: 1.2862, Train Acc: 0.5000 | Val Loss: 1.2938, Val Acc: 0.5020
Epoch 8/20 | Train Loss: 1.2418, Train Acc: 0.5162 | Val Loss: 1.2486, Val Acc: 0.5164
Epoch 9/20 | Train Loss: 1.2068, Train Acc: 0.5292 | Val Loss: 1.1923, Val Acc: 0.5440
Epoch 10/20 | Train Loss: 1.1721, Train Acc: 0.5413 | Val Loss: 1.1813, Val Acc: 0.5526
Epoch 11/20 | Train Loss: 1.1425, Train Acc: 0.5543 | Val Loss: 1.2287, Val Acc: 0.5428
Epoch 12/20 | Train Loss: 1.0942, Train A

epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
train_acc,▁▃▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train_loss,█▆▅▅▅▄▄▄▄▃▃▃▃▂▂▂▂▁▁▁
val_acc,▁▄▄▆▆▇▆▆▇█▇█▅▇▇█████
val_loss,█▅▄▃▂▂▃▂▁▁▂▁▄▂▂▂▁▂▂▂
epoch,20
train_acc,0.67395
train_loss,0.82246
val_acc,0.56629
val_loss,1.26778


In [16]:
config_a2_r3 = {"architecture": "DeeperCNN", "learning_rate": 0.001, "batch_size": 64,
                 "epochs": 20, "optimizer": "Adam", "dropout": 0.5, "weight_decay": 1e-4}

model_a2_r3 = DeeperCNN(dropout_p=0.5)
model_a2_r3 = train_model(model_a2_r3, train_loader, val_loader, config_a2_r3,
                           group_name="architecture-2-deeper-cnn",
                           run_name="weight_decay-1e-4")

Epoch 1/20 | Train Loss: 1.8143, Train Acc: 0.2780 | Val Loss: 1.6371, Val Acc: 0.3381
Epoch 2/20 | Train Loss: 1.5788, Train Acc: 0.3785 | Val Loss: 1.4560, Val Acc: 0.4307
Epoch 3/20 | Train Loss: 1.4834, Train Acc: 0.4205 | Val Loss: 1.4474, Val Acc: 0.4370
Epoch 4/20 | Train Loss: 1.4265, Train Acc: 0.4450 | Val Loss: 1.3315, Val Acc: 0.4783
Epoch 5/20 | Train Loss: 1.3717, Train Acc: 0.4698 | Val Loss: 1.3275, Val Acc: 0.4897
Epoch 6/20 | Train Loss: 1.3289, Train Acc: 0.4852 | Val Loss: 1.2728, Val Acc: 0.5103
Epoch 7/20 | Train Loss: 1.2970, Train Acc: 0.4961 | Val Loss: 1.2628, Val Acc: 0.5085
Epoch 8/20 | Train Loss: 1.2675, Train Acc: 0.5095 | Val Loss: 1.2494, Val Acc: 0.5161
Epoch 9/20 | Train Loss: 1.2213, Train Acc: 0.5272 | Val Loss: 1.2086, Val Acc: 0.5347
Epoch 10/20 | Train Loss: 1.1882, Train Acc: 0.5369 | Val Loss: 1.2228, Val Acc: 0.5294
Epoch 11/20 | Train Loss: 1.1607, Train Acc: 0.5463 | Val Loss: 1.1956, Val Acc: 0.5489
Epoch 12/20 | Train Loss: 1.1106, Train A

epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
train_acc,▁▃▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train_loss,█▆▆▅▅▅▄▄▄▄▃▃▃▃▂▂▂▁▁▁
val_acc,▁▄▄▅▆▆▆▇▇▇█▇█▇█▇█▇██
val_loss,█▅▅▃▃▂▂▂▁▂▁▁▁▂▂▂▂▂▃▅
epoch,20
train_acc,0.66917
train_loss,0.83204
val_acc,0.54957
val_loss,1.41482


In [17]:
config_a2_r4 = {
    "architecture": "DeeperCNN",
    "learning_rate": 0.0005,
    "batch_size": 64,
    "epochs": 20,
    "optimizer": "Adam",
    "dropout": 0.3,
    "weight_decay": 0
}

model_a2_r4 = DeeperCNN(dropout_p=0.3)
model_a2_r4 = train_model(model_a2_r4, train_loader, val_loader, config_a2_r4,
                           group_name="architecture-2-deeper-cnn",
                           run_name="dropout-0.3_lr-0.0005")

Epoch 1/20 | Train Loss: 1.6509, Train Acc: 0.3512 | Val Loss: 1.4826, Val Acc: 0.4270
Epoch 2/20 | Train Loss: 1.4027, Train Acc: 0.4583 | Val Loss: 1.3427, Val Acc: 0.4813
Epoch 3/20 | Train Loss: 1.2965, Train Acc: 0.5037 | Val Loss: 1.2483, Val Acc: 0.5124
Epoch 4/20 | Train Loss: 1.2264, Train Acc: 0.5340 | Val Loss: 1.2050, Val Acc: 0.5326
Epoch 5/20 | Train Loss: 1.1594, Train Acc: 0.5567 | Val Loss: 1.2810, Val Acc: 0.4885
Epoch 6/20 | Train Loss: 1.1017, Train Acc: 0.5826 | Val Loss: 1.1793, Val Acc: 0.5514
Epoch 7/20 | Train Loss: 1.0491, Train Acc: 0.6018 | Val Loss: 1.1428, Val Acc: 0.5579
Epoch 8/20 | Train Loss: 1.0002, Train Acc: 0.6192 | Val Loss: 1.1463, Val Acc: 0.5663
Epoch 9/20 | Train Loss: 0.9321, Train Acc: 0.6463 | Val Loss: 1.1830, Val Acc: 0.5500
Epoch 10/20 | Train Loss: 0.8693, Train Acc: 0.6672 | Val Loss: 1.2230, Val Acc: 0.5468
Epoch 11/20 | Train Loss: 0.8103, Train Acc: 0.6914 | Val Loss: 1.1621, Val Acc: 0.5844
Epoch 12/20 | Train Loss: 0.7420, Train A

epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
train_acc,▁▂▃▃▄▄▄▅▅▅▆▆▆▇▇▇▇███
train_loss,█▇▆▆▅▅▅▅▄▄▃▃▃▂▂▂▂▁▁▁
val_acc,▁▃▅▆▄▇▇▇▆▆███▇█▇▇███
val_loss,▆▄▃▂▃▂▁▁▂▂▁▁▃▄▃▄▅▅▇█
epoch,20
train_acc,0.87071
train_loss,0.34563
val_acc,0.57581
val_loss,1.61922


In [18]:
config_a2_r5 = {
    "architecture": "DeeperCNN",
    "learning_rate": 0.0005,
    "batch_size": 64,
    "epochs": 20,
    "optimizer": "Adam",
    "dropout": 0.0,
    "weight_decay": 1e-4
}

model_a2_r5 = DeeperCNN(dropout_p=0.0)
model_a2_r5 = train_model(model_a2_r5, train_loader, val_loader, config_a2_r5,
                           group_name="architecture-2-deeper-cnn",
                           run_name="no-dropout_weight_decay-1e-4")

Epoch 1/20 | Train Loss: 1.6243, Train Acc: 0.3673 | Val Loss: 1.4779, Val Acc: 0.4335
Epoch 2/20 | Train Loss: 1.3348, Train Acc: 0.4902 | Val Loss: 1.3176, Val Acc: 0.4969
Epoch 3/20 | Train Loss: 1.2261, Train Acc: 0.5351 | Val Loss: 1.2211, Val Acc: 0.5363
Epoch 4/20 | Train Loss: 1.1422, Train Acc: 0.5674 | Val Loss: 1.2174, Val Acc: 0.5340
Epoch 5/20 | Train Loss: 1.0510, Train Acc: 0.6046 | Val Loss: 1.2252, Val Acc: 0.5435
Epoch 6/20 | Train Loss: 0.9802, Train Acc: 0.6305 | Val Loss: 1.2292, Val Acc: 0.5370
Epoch 7/20 | Train Loss: 0.8889, Train Acc: 0.6715 | Val Loss: 1.1829, Val Acc: 0.5603
Epoch 8/20 | Train Loss: 0.8085, Train Acc: 0.7009 | Val Loss: 1.2060, Val Acc: 0.5526
Epoch 9/20 | Train Loss: 0.7090, Train Acc: 0.7384 | Val Loss: 1.2928, Val Acc: 0.5551
Epoch 10/20 | Train Loss: 0.6204, Train Acc: 0.7718 | Val Loss: 1.2607, Val Acc: 0.5674
Epoch 11/20 | Train Loss: 0.5067, Train Acc: 0.8209 | Val Loss: 1.3131, Val Acc: 0.5735
Epoch 12/20 | Train Loss: 0.4277, Train A

epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
train_acc,▁▂▃▃▄▄▅▅▅▆▆▇▇▇██████
train_loss,█▇▆▆▅▅▅▄▄▃▃▃▂▂▂▁▁▁▁▁
val_acc,▁▄▆▆▇▆▇▇▇███▇▇█▇▇█▇▇
val_loss,▃▂▁▁▁▁▁▁▂▁▂▂▃▄▄▅▆▇▇█
epoch,20
train_acc,0.97422
train_loss,0.09651
val_acc,0.54795
val_loss,2.28349


In [21]:
config_a2_r6 = {
    "architecture": "DeeperCNN",
    "learning_rate": 0.0005,
    "batch_size": 64,
    "epochs": 25,
    "optimizer": "Adam",
    "dropout": 0.3,
    "weight_decay": 0,
    "scheduler": "ReduceLROnPlateau",
    "early_stopping": True,
    "early_stop_patience": 4
}

model_a2_r6 = DeeperCNN(dropout_p=0.3)
model_a2_r6 = train_model(
    model_a2_r6, train_loader, val_loader, config_a2_r6,
    group_name="architecture-2-deeper-cnn",
    run_name="adam-0.0005_scheduler_early_stop"
)

Epoch 1/25 | Train Loss: 1.6938, Train Acc: 0.3322 | Val Loss: 1.5136, Val Acc: 0.4130
Epoch 2/25 | Train Loss: 1.4400, Train Acc: 0.4466 | Val Loss: 1.4141, Val Acc: 0.4414
Epoch 3/25 | Train Loss: 1.3304, Train Acc: 0.4920 | Val Loss: 1.2976, Val Acc: 0.4985
Epoch 4/25 | Train Loss: 1.2535, Train Acc: 0.5242 | Val Loss: 1.2452, Val Acc: 0.5222
Epoch 5/25 | Train Loss: 1.1869, Train Acc: 0.5550 | Val Loss: 1.1980, Val Acc: 0.5331
Epoch 6/25 | Train Loss: 1.1282, Train Acc: 0.5717 | Val Loss: 1.1967, Val Acc: 0.5370
Epoch 7/25 | Train Loss: 1.0771, Train Acc: 0.5902 | Val Loss: 1.1605, Val Acc: 0.5584
Epoch 8/25 | Train Loss: 1.0195, Train Acc: 0.6095 | Val Loss: 1.1788, Val Acc: 0.5449
Epoch 9/25 | Train Loss: 0.9565, Train Acc: 0.6358 | Val Loss: 1.2155, Val Acc: 0.5544
Epoch 10/25 | Train Loss: 0.9028, Train Acc: 0.6567 | Val Loss: 1.1616, Val Acc: 0.5651
Epoch 11/25 | Train Loss: 0.7594, Train Acc: 0.7110 | Val Loss: 1.1733, Val Acc: 0.5800


epoch,▁▂▂▃▄▅▅▆▇▇█
learning_rate,██████████▁
train_acc,▁▃▄▅▅▅▆▆▇▇█
train_loss,█▆▅▅▄▄▃▃▂▂▁
val_acc,▁▂▅▆▆▆▇▇▇▇█
val_loss,█▆▄▃▂▂▁▁▂▁▁
epoch,11
learning_rate,0.00025
train_acc,0.71102
train_loss,0.75942
val_acc,0.57999
